<a href="https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hawaekrami/week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I'm predicting is_declining_label (impressions dropped >20% from Feb to March 2026)
a yes/no question with a real observed outcome, so per the toolkit I'm starting with
Logistic Regression: readable, coefficients are explainable in plain language, and its
predicted probability doubles as a ranking score for precision@K (my lane is a
"which pages first" ranking question, and probability-based ranking is exactly what
that needs). I'll only escalate to Random Forest if Logistic Regression clearly
underfits the pattern.

In [3]:
%pip -q install duckdb scikit-learn
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FEB = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')"
MAR = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

# Features: February only (strictly BEFORE the label period)
feb = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        ROUND(100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0), 2) AS ctr_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM {FEB}
    GROUP BY content_hash_id, client_hash_id
""").df()

# Label source: March impressions, to compare against February
mar = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_mar
    FROM {MAR}
    GROUP BY content_hash_id
""").df()

data = feb.merge(mar, on="content_hash_id", how="left")
data["impressions_mar"] = data["impressions_mar"].fillna(0)

# trend_pct, mirroring the starter CSV's own definition (blank/0-prev -> 0, not divide-by-zero)
data["trend_pct"] = 0.0
nonzero_prev = data["impressions_feb"] > 0
data.loc[nonzero_prev, "trend_pct"] = (
    100.0 * (data.loc[nonzero_prev, "impressions_mar"] - data.loc[nonzero_prev, "impressions_feb"])
    / data.loc[nonzero_prev, "impressions_feb"]
)

data["is_declining_label"] = (data["trend_pct"] < -20).astype(int)

print("Total pages:", len(data))
print("Declining rate (base rate):", round(data["is_declining_label"].mean() * 100, 1), "%")
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages: 321546
Declining rate (base rate): 14.4 %


,content_hash_id,client_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,impressions_mar,trend_pct,is_declining_label
0,content_b1fc2cbd0eb808db,client_3ffa76342f366962,0.0,0.0,NaN,NaN,0.0,0.0,0
1,content_fb84747a57b8b665,client_3ffa76342f366962,0.0,0.0,NaN,NaN,0.0,0.0,0
2,content_feccf822ac21326e,client_3ffa76342f366962,0.0,0.0,NaN,NaN,0.0,0.0,0
3,content_43147be54c74d162,client_3ffa76342f366962,0.0,0.0,NaN,NaN,0.0,0.0,0
4,content_48995646c9f4fb7a,client_3ffa76342f366962,0.0,0.0,NaN,NaN,0.0,0.0,0


## 2. Split design

Two honest constraints, combined: (1) Time-aware every feature (impressions_feb,
clicks_feb, ctr_feb, avg_position_feb) is measured in February, strictly before the
label period (Feb-to-March trend), so the model never sees the future it's predicting.
(2) Grouped by client — I split on client_hash_id, not on individual rows, so every
page from a given client lands entirely in train or entirely in test. This matters
because pages from the same client tend to share client-specific baseline behavior
(their SEO team, their content style); a random row split would leak that shared
client pattern between train and test and make the score look better than it really is.

In [4]:
import numpy as np

rng = np.random.default_rng(42)  # fixed seed, so this split is reproducible

clients = data["client_hash_id"].unique()
rng.shuffle(clients)

n_test = int(len(clients) * 0.2)
test_clients = set(clients[:n_test])
train_clients = set(clients[n_test:])

train = data[data["client_hash_id"].isin(train_clients)].copy()
test = data[data["client_hash_id"].isin(test_clients)].copy()

print("Train clients:", len(train_clients), "| Train rows:", len(train))
print("Test clients:", len(test_clients), "| Test rows:", len(test))
print("Train decline rate:", round(train["is_declining_label"].mean() * 100, 1), "%")
print("Test decline rate:", round(test["is_declining_label"].mean() * 100, 1), "%")


Train clients: 44 | Train rows: 289717
Test clients: 10 | Test rows: 31829
Train decline rate: 13.3 %
Test decline rate: 23.9 %


## 3. Train + compare vs my baseline

Same test set, same metric (precision@50), for both: my Week-4 rule (recomputed on
February features, so it's judged on the same information the model gets) and a
Logistic Regression trained on the train clients only.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

features = ["impressions_feb", "clicks_feb", "ctr_feb", "avg_position_feb"]

X_train = train[features].fillna(0)
y_train = train["is_declining_label"]
X_test = test[features].fillna(0)
y_test = test["is_declining_label"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)

test = test.copy()
test["model_score"] = model.predict_proba(X_test_scaled)[:, 1]
# Volume floor: don't trust the model's score on rows with almost no real traffic
# (single-impression rows can produce artificially extreme position/CTR values)
MIN_IMPRESSIONS = 500
test.loc[test["impressions_feb"] < MIN_IMPRESSIONS, "model_score"] = 0.0

# Recompute the Week-4 rule on February features, same eligibility/logic as before
CTR_BENCHMARK = 0.5
eligible = (test["impressions_feb"] >= 500) & (test["avg_position_feb"] > 0) & (test["avg_position_feb"] <= 20)
ctr_gap = (CTR_BENCHMARK - test["ctr_feb"]).clip(lower=0)
test["baseline_score"] = 0.0
test.loc[eligible, "baseline_score"] = test.loc[eligible, "impressions_feb"] * ctr_gap[eligible]

def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k["is_declining_label"].mean() * 100

base_rate = y_test.mean() * 100
baseline_p50 = precision_at_k(test, "baseline_score", 50)
model_p50 = precision_at_k(test, "model_score", 50)

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 baseline rule", "Logistic Regression"],
    "precision@50": [round(base_rate, 1), round(baseline_p50, 1), round(model_p50, 1)]
})
print(comparison.to_string(index=False))


              method  precision@50
  Base rate (random)          23.9
Week-4 baseline rule          58.0
 Logistic Regression          84.0


## 4. Errors and interpretation
What it leans on: avg_position_feb dominates (coefficient 0.71, ~4x the next feature),
followed by ctr_feb (0.18) and impressions_feb (0.10). This is directionally sensible —
worse position and lower CTR should both predict decline — but the size of the position
coefficient made it worth stress-testing, which paid off (see below).

What went wrong first, and the fix: at a 30-impression floor, 21 of the top 50 (42%) were
false positives, and the 3 inspected cases all showed impossible trend_pct values (up to
+7093%) alongside oddly extreme avg_position_feb (~80-86) — a strong sign of noisy,
low-history position data rather than genuinely low-quality pages. Raising the volume
floor to 500 (matching the Week-4 baseline's own threshold) cut false positives to 8 of
50 (16%) and removed the impossible trend values entirely. This was a real, measurable
improvement, not just guesswork — the fix is documented here rather than silently applied.

What's still wrong: the remaining false positives (e.g. content_2544b0336dd4184a,
trend_pct = +52.6%) are pages the model expected to decline that instead grew. These look
like genuine misses, not data artifacts — the model's current features (Feb position/CTR/
volume alone) can't distinguish "about to

In [11]:
# What does the model lean on?
import pandas as pd
coef_table = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print("--- Feature coefficients (standardized, so comparable) ---")
print(coef_table.to_string(index=False))

# Top 50 by model score: how many wrong, and what do the wrong ones look like?
top50 = test.sort_values("model_score", ascending=False).head(50).copy()
wrong = top50[top50["is_declining_label"] == 0]

print(f"\nOf the top 50 by model score, {len(wrong)} were NOT actually declining (false positives).")
print("\n--- 3 concrete wrong cases ---")
print(wrong[["content_hash_id", "impressions_feb", "clicks_feb", "ctr_feb",
             "avg_position_feb", "trend_pct", "model_score"]].head(3).to_string(index=False))


--- Feature coefficients (standardized, so comparable) ---
         feature  coefficient
avg_position_feb     0.713959
         ctr_feb     0.175948
 impressions_feb     0.100774
      clicks_feb    -0.053948

Of the top 50 by model score, 8 were NOT actually declining (false positives).

--- 3 concrete wrong cases ---
         content_hash_id  impressions_feb  clicks_feb  ctr_feb  avg_position_feb  trend_pct  model_score
content_2544b0336dd4184a            680.0         0.0      0.0         73.660483  52.647059     0.984332
content_743afd9c6979d5aa           1041.0         1.0      0.1         68.942925  74.063401     0.979353
content_4386a049011ab551            739.0         0.0      0.0         64.000994  67.117727     0.971708


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.